In [9]:
!pip install radon textstat google-genai ipywidgets pandas

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google import genai
import radon.complexity as rc
import textstat
import difflib
import pandas as pd
from datetime import datetime

# 1. Initialize Gemini Client (Replace with your actual key string)
API_KEY = os.environ.get("GEMINI_API_KEY")client = genai.Client(api_key=API_KEY)
# Global memory to hold auditing logs
analysis_history = []

# =====================================================================
# CORE UTILITY METRIC PIPELINES
# =====================================================================

def get_complexity_score(code):
    """Calculate cyclomatic complexity — higher = harder to maintain"""
    try:
        results = rc.cc_visit(code)
        if not results:
            return 1, "Low", "🟢"
        avg = sum(r.complexity for r in results) / len(results)
        if avg <= 5:
            return round(avg, 1), "Low", "🟢"
        elif avg <= 10:
            return round(avg, 1), "Medium", "🟡"
        else:
            return round(avg, 1), "High", "🔴"
    except:
        return 0, "Unknown", "⚪"

def display_diff_html(original, fixed):
    """Generates and displays a clean colored HTML structural diff block"""
    diff = list(difflib.unified_diff(
        original.splitlines(keepends=True),
        fixed.splitlines(keepends=True),
        fromfile='Original',
        tofile='AI Fixed',
        lineterm=''
    ))
    
    html_lines = []
    for line in diff:
        if line.startswith('+') and not line.startswith('+++'):
            html_lines.append(f'<div style="background:#1a4a1a;color:#90ee90;padding:2px 8px;font-family:monospace;white-space:pre">{line}</div>')
        elif line.startswith('-') and not line.startswith('---'):
            html_lines.append(f'<div style="background:#4a1a1a;color:#ff9090;padding:2px 8px;font-family:monospace;white-space:pre">{line}</div>')
        elif not line.startswith('@@') and not line.startswith('---') and not line.startswith('+++'):
            html_lines.append(f'<div style="background:#1e1e1e;color:#ccc;padding:2px 8px;font-family:monospace;white-space:pre">{line}</div>')
    
    if html_lines:
        display(HTML('<h4 style="color:#4DB6AC; margin-top:15px;">🛠️ Code Structural Changes (Diff View):</h4>'))
        display(HTML('<div style="border:1px solid #444;border-radius:8px;overflow:hidden;margin-bottom:15px;">' + ''.join(html_lines) + '</div>'))

def show_history_table():
    """Renders the session auditing tracking table inside the notebook"""
    if not analysis_history:
        return
    df = pd.DataFrame(analysis_history)
    display(HTML('<h4 style="color:#4DB6AC; margin-top:15px;">📋 Session Audit History Trail:</h4>'))
    display(df.style.set_properties(**{
        'background-color': '#1e1e1e',
        'color': '#e0e0e0',
        'border': '1px solid #444',
        'padding': '6px 12px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#333'), ('color', '#4DB6AC'), ('font-weight', 'bold')]
    }]))

# =====================================================================
# INTERACTIVE UI WIDGET DESIGN
# =====================================================================

code_input = widgets.Textarea(
    value='def process_data(x):\n    if x > 10:\n        return x * 2\n    else:\n        return x + 5',
    placeholder='Paste your code snippet here...',
    description='Code window:',
    layout=widgets.Layout(width='95%', height='180px')
)

task_dropdown = widgets.Dropdown(
    options=['Explain Line-by-Line', 'Find Bugs & Fix', 'Optimize Performance'],
    value='Explain Line-by-Line',
    description='AI Strategy:',
    layout=widgets.Layout(width='40%')
)

run_button = widgets.Button(
    description='Run AI Code Audit ✨',
    button_style='success',
    layout=widgets.Layout(width='40%', margin='0px 0px 0px 10px')
)

# Output panel containment window
output_panel = widgets.Output()

# =====================================================================
# REACTIVE LOGIC BUTTON CLICK HANDLER
# =====================================================================

def on_button_clicked(b):
    with output_panel:
        clear_output(wait=True)
        
        raw_code = code_input.value.strip()
        selected_task = task_dropdown.value
        
        if not raw_code:
            display(HTML("<b style='color:#ff9090;'>❌ Error: Code input field cannot be left completely blank.</b>"))
            return
            
        print("🔄 Connecting to Gemini Processing Engine...")
        
        # 1. Run local computational complexity engine instantly
        score, rating, icon = get_complexity_score(raw_code)
        display(HTML(f"<div style='background:#2a2a2a; padding:10px; border-radius:6px; border-left:5px solid #4DB6AC;'><h4>Complexity Score: {icon} {score} ({rating} Maintenance Risk)</h4></div>"))
        
        # 2. Frame specific targeted developer prompt structures using bulletproof triple-quotes
        if selected_task == 'Explain Line-by-Line':
            prompt = f"""Perform an explicit line-by-line breakdown explaining how this code functions structurally. Keep it structured and easy to read:

{raw_code}"""
        elif selected_task == 'Find Bugs & Fix':
            prompt = f"""Audit this code for errors, memory leaks, or bad practices. Return the updated clean code wrapped inside markdown ```python blocks, followed by an explanation of the adjustments made:

{raw_code}"""
        else:
            prompt = f"""Optimize this code for execution speed and space efficiency. Return the optimized code snippet wrapped inside markdown ```python blocks, followed by complexity analysis explanations:

{raw_code}"""
            
        try:
            # 3. Stream data from model
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt
            )
            ai_output = response.text
            
            # Print text response from the AI
            print("\n💡 AI ANALYSIS REPORT:")
            print(ai_output)
            print("-" * 80)
            
            # 4. Extract generated clean code snippets for the custom Diff engine
            if selected_task in ['Find Bugs & Fix', 'Optimize Performance']:
                cleaned_ai_code = ""
                if "```python" in ai_output:
                    cleaned_ai_code = ai_output.split("```python")[1].split("```")[0].strip()
                elif "```" in ai_output:
                    cleaned_ai_code = ai_output.split("```")[1].split("```")[0].strip()
                
                # Render the structural diff layout comparison if extraction succeeds
                if cleaned_ai_code and cleaned_ai_code != raw_code: 
                    display_diff_html(raw_code, cleaned_ai_code)
            
            # 5. Append structural session logs to local pandas memory array
            analysis_history.append({
                'Time': datetime.now().strftime('%H:%M:%S'),
                'Task': selected_task,
                'Code Preview': raw_code[:40] + '...' if len(raw_code) > 40 else raw_code,
                'Complexity': f"{score} ({rating})",
                'Result Preview': ai_output[:60] + '...'
            })
            
            # 6. Render the updated history log table component at the bottom
            show_history_table()
            
        except Exception as e:
            display(HTML(f"<b style='color:#ff9090;'>🚨 API Execution Failure: {str(e)}</b>"))

# Link button element to the tracking runtime logic
run_button.on_click(on_button_clicked)

# =====================================================================
# RENDER INTEGRATED LAYOUT
# =====================================================================
display(HTML("<h2 style='color:#4DB6AC; margin-bottom:10px;'>🛠️ Professional AI Code Audit Dashboard</h2>"))
display(code_input)
display(widgets.HBox([task_dropdown, run_button], layout=widgets.Layout(margin='10px 0px')))
display(output_panel)

Textarea(value='def process_data(x):\n    if x > 10:\n        return x * 2\n    else:\n        return x + 5', …

Output()

In [2]:
# ── MODEL EVALUATION — run this after testing several code samples ──
test_cases = [
    {
        "code": "def div(a,b): return a/b",
        "expected_issues": ["No zero division check"],
        "task": "Find Bugs & Fix"
    },
    {
        "code": "for i in range(len(arr)): print(arr[i])",
        "expected_issues": ["Should use enumerate or direct iteration"],
        "task": "Optimize Performance"
    },
]

print("=" * 60)
print("MODEL EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash")
print(f"Test cases: {len(test_cases)}")
print(f"Tasks evaluated: Code Fix, Optimization")
print(f"Evaluation method: Manual review against expected outputs")
print("=" * 60)

# Run each test case
for i, test in enumerate(test_cases, 1):
    score, level, emoji = get_complexity_score(test['code'])
    print(f"\nTest {i}: {test['task']}")
    print(f"  Code: {test['code']}")
    print(f"  Complexity: {emoji} {level} ({score})")
    print(f"  Expected issues: {test['expected_issues']}")
    print(f"  Status: Run manually and verify AI caught expected issues")

MODEL EVALUATION SUMMARY
Model: Gemini 2.5 Flash
Test cases: 2
Tasks evaluated: Code Fix, Optimization
Evaluation method: Manual review against expected outputs

Test 1: Find Bugs & Fix
  Code: def div(a,b): return a/b
  Complexity: 🟢 Low (1.0)
  Expected issues: ['No zero division check']
  Status: Run manually and verify AI caught expected issues

Test 2: Optimize Performance
  Code: for i in range(len(arr)): print(arr[i])
  Complexity: 🟢 Low (1)
  Expected issues: ['Should use enumerate or direct iteration']
  Status: Run manually and verify AI caught expected issues
